# 14. CLA 因果層別分析（Causal Layered Analysis） — 練習問題

**対象技術**: AI（生成AI・機械学習・自動化を含む）

因果層別分析（CLA）は、ある技術をめぐる言説を深さの異なる4層 — litany（表層の出来事・統計）/ system（構造的原因）/ worldview（世界観・パラダイム）/ myth（深層の物語・隠喩）— に分けて読み解く手法である。言説がどの層に偏っているかを見ることで、議論が表層に留まっているか深層に届いているかを診断する。このノートブックではAIをめぐる言説を題材にする。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


## 4層とキーワード辞書の定義

CLA の4層と、各層を特徴づけるキーワード辞書を定義する。キーワード一致数で言説を機械分類するための辞書である。

> **位置づけの注記**: 以下のキーワード辞書による層別自動分類は、litany 層の語彙の偏りを可視化するための補助にすぎない。worldview と myth は語彙ではなく解釈を要する深層であり、機械分類できない（解説 `methods/14_causal_layered_analysis.md` 第6節）。本ノートブックでは、このセクションのあとに CLA の核心である(A) 4層の構造化と myth 置換による代替未来構築、(B) 層間支持関係マッピング、(C) 分類器の限界の定量化、を追加する。

In [ ]:
LAYERS = ["litany", "system", "worldview", "myth"]
LAYER_JP = {
    "litany": "litany(表層の出来事)",
    "system": "system(構造的原因)",
    "worldview": "worldview(世界観)",
    "myth": "myth(深層の隠喩)",
}

LAYER_KEYWORDS = {
    "litany": [
        "見出し", "報道", "ニュース", "統計", "失業", "速報",
        "発表", "話題", "ランキング", "事故", "炎上",
    ],
    "system": [
        "労働市場", "教育制度", "プラットフォーム", "データ独占", "規制",
        "雇用構造", "政策", "投資", "制度", "競争", "サプライチェーン",
    ],
    "worldview": [
        "効率", "技術決定論", "パラダイム", "進歩", "知能", "計算",
        "前提", "価値観", "合理性", "生産性至上", "中立",
    ],
    "myth": [
        "機械の知性", "魔法", "神", "物語", "夢", "悪夢",
        "人間 vs 機械", "革命", "黙示録", "自動化の魔法", "知性",
    ],
}

## 分析対象の言説コーパス

AIの社会的インパクトをめぐる言説の例を用意する。

In [ ]:
DISCOURSE = [
    "AIが仕事を奪うというニュースの見出しが連日報道されている",
    "生成AIが嘘をつくという話題と、誤回答の統計が速報で流れた",
    "AIの影響は労働市場の雇用構造と教育制度に依存して現れる",
    "プラットフォーム企業のデータ独占と規制の遅れが競争を歪めている",
    "効率こそ善だという生産性至上の技術決定論が議論の前提になっている",
    "知能とは計算であるというパラダイムが、AIへの期待の価値観を支える",
    "機械の知性という隠喩が、人間 vs 機械という物語を人々に呼び起こす",
    "AIという神の物語が、自動化の魔法への夢と失業の悪夢を同時に育てる",
    "AI人材の投資と政策をめぐる制度設計の議論が進む",
    "AIが社会を変える革命だという神話が研究投資の物語を支えている",
]

print(f"言説コーパス: {len(DISCOURSE)} 件")

## 言説の層別自動分類

各言説を、4層それぞれのキーワード一致数で採点し、最も多い層に分類する。

In [ ]:
def classify_discourse(text, keyword_dict):
    """1つの言説を各層のキーワード一致数で採点し最も多い層に分類する。
    戻り値: (判定された層, 各層のスコア辞書)。
    """
    scores = {layer: 0 for layer in LAYERS}
    for layer, words in keyword_dict.items():
        for w in words:
            if w in text:
                scores[layer] += 1
    best_layer = max(scores, key=scores.get)
    if scores[best_layer] == 0:
        return "unclassified", scores
    return best_layer, scores


layer_counts = {l: 0 for l in LAYERS}
layer_counts["unclassified"] = 0

print("[言説の層別自動分類]")
print("-" * 70)
for i, text in enumerate(DISCOURSE, start=1):
    layer, scores = classify_discourse(text, LAYER_KEYWORDS)
    layer_counts[layer] += 1
    label = LAYER_JP.get(layer, "未分類")
    print(f"  言説{i:>2} [{label}]")
    print(f"         「{text}」")

## 層別分布の集計と深さの偏り診断

層別の出現分布を集計し、議論が表層（litany+system）寄りか深層（worldview+myth）寄りかを診断する。

In [ ]:
print("[層別の出現分布]")
print("-" * 70)
total = len(DISCOURSE)
for l in LAYERS:
    cnt = layer_counts[l]
    bar = "#" * cnt
    pct = 100 * cnt / total
    print(f"  {LAYER_JP[l]:<22} {cnt:>2}件 ({pct:>4.0f}%) {bar}")
if layer_counts["unclassified"]:
    print(f"  未分類                 {layer_counts['unclassified']:>2}件")

shallow = layer_counts["litany"] + layer_counts["system"]
deep = layer_counts["worldview"] + layer_counts["myth"]
print("-" * 70)
print(f"  表層寄り(litany+system) = {shallow}件 / "
      f"深層寄り(worldview+myth) = {deep}件")
if shallow > deep:
    print("  -> 議論は表層(出来事と構造)に偏る。worldview と myth を")
    print("     掘り下げ、隠喩を変えた代替未来の構築が次の課題。")
elif deep > shallow:
    print("  -> 議論は深層(世界観と隠喩)まで到達している。")
    print("     代替未来の構築に進む素地がある。")
else:
    print("  -> 表層と深層が均衡。4層を貫く再構成に取り組める。")

## 可視化: 4層別の言説出現分布

litany / system / worldview / myth の各層の言説件数を棒グラフで描く。浅い層ほど明るく、深い層ほど濃い色にして、表層偏重か深層到達かが視覚的に分かるようにする。あわせて表層計と深層計を注記する。

In [ ]:
labels = ["litany", "system", "worldview", "myth"]
values = [layer_counts[l] for l in LAYERS]
colors = ["#cfe8f3", "#7fb8d6", "#3f7ca6", "#1f3f5c"]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(range(len(LAYERS)), values, color=colors)
ax.set_xticks(range(len(LAYERS)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel("Number of discourse statements")
ax.set_title("CLA: depth distribution of AI discourse")
for i, v in enumerate(values):
    ax.text(i, v + 0.05, str(v), ha="center", fontsize=10)

# 表層 / 深層 の区切りと注記
ax.axvline(1.5, color="#c0504d", linestyle="--", linewidth=1)
ax.text(0.5, max(values) * 0.92, f"shallow = {shallow}",
        ha="center", fontsize=9, color="#c0504d")
ax.text(2.5, max(values) * 0.92, f"deep = {deep}",
        ha="center", fontsize=9, color="#c0504d")
fig.tight_layout()
plt.show()

## (A) 4層の構造化表現と myth 置換による代替未来の構築

CLA の核心は、技術をめぐる支配的フレーミングを4層の構造化データとして明示し、最深層の myth/メタファーを意図的に別の隠喩へ置換することにある。myth を置き換えると、それを土台とする worldview → system → litany が連動して書き換わる。ここでは「現状フレーミング」と、myth を置換した「代替フレーミング」を4層そろえて並べ、同じ技術でも最深層の物語を変えるだけでインパクトの像が反転することを確認する（手法手順5・適用例）。

In [ ]:
# --- AIをめぐる現状フレーミングを4層の構造化データで保持する ---
# 各層は複数要素のリスト。深い層ほど解釈的で、語彙では拾えない。
CURRENT_FRAME = {
    "myth": [
        "機械の知性",
        "人間 vs 機械",
        "魔法のような自動化",
    ],
    "worldview": [
        "効率至上主義",
        "技術決定論",
        "知能＝計算",
    ],
    "system": [
        "労働市場の不安定化",
        "教育制度の追従の遅れ",
        "プラットフォーム経済とデータ独占",
        "規制の遅れ",
    ],
    "litany": [
        "AIが仕事を奪うという見出し報道",
        "AIが嘘をつくという速報",
    ],
}

# --- 代替フレーミング: 最深層 myth を置換し、上層を連動して書き換える ---
ALT_FRAME = {
    "myth": [
        "AI＝協働の道具",
    ],
    "worldview": [
        "知能は人と道具の協働で立ち上がる",
        "技術は選び取れる(技術決定論の否定)",
    ],
    "system": [
        "AIリテラシー教育の制度化",
        "労働移行支援の仕組み",
        "データガバナンスと共有の設計",
    ],
    "litany": [
        "AIで定型業務が減り創造と余暇に時間が回る",
    ],
}

LAYER_ORDER = ["litany", "system", "worldview", "myth"]
print("[4層の構造化表現を定義した]")
for fname, frame in [("現状フレーミング", CURRENT_FRAME),
                     ("代替フレーミング", ALT_FRAME)]:
    n = sum(len(frame[l]) for l in LAYER_ORDER)
    print(f"  {fname}: 全{n}要素")


In [ ]:
# --- myth 置換: myth を起点に worldview->system->litany が連動する対応 ---
# 置換ルールを明示し、現状と代替を4層そろえて並べて出力する。
SUBSTITUTION = {
    "myth": ("機械の知性 / 人間 vs 機械 / 魔法のような自動化",
             "AI＝協働の道具"),
    "worldview": ("効率至上主義・技術決定論・知能＝計算",
                  "知能は人と道具の協働で立ち上がる"),
    "system": ("労働市場・データ独占・規制の遅れ",
               "AIリテラシー教育・労働移行支援・データガバナンス"),
    "litany": ("AIが仕事を奪う / AIが嘘をつく",
               "AIで定型業務が減り創造と余暇に時間が回る"),
}

def print_frame(title, frame):
    print(title)
    print("-" * 64)
    for layer in ["myth", "worldview", "system", "litany"]:
        print(f"  [{layer}]")
        for item in frame[layer]:
            print(f"      - {item}")
    print()

print_frame("=== 現状フレーミング (支配的) ===", CURRENT_FRAME)
print("        v  最深層 myth を置換  v")
print("        v  連動して上層が書き換わる  v\n")
print_frame("=== 代替フレーミング (myth 置換後) ===", ALT_FRAME)

print("[myth 置換による4層の連動]")
print("-" * 64)
for layer in ["myth", "worldview", "system", "litany"]:
    cur, alt = SUBSTITUTION[layer]
    print(f"  {layer:>9}:  {cur}")
    print(f"  {'':>9}   ->  {alt}\n")


In [ ]:
# --- 現状/代替フレーミングを4層対比で可視化する ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
frames = [("Current frame", CURRENT_FRAME, "#c0504d"),
          ("Alternative frame (myth substituted)", ALT_FRAME, "#3f7ca6")]
layer_y = {"myth": 0, "worldview": 1, "system": 2, "litany": 3}

for ax, (title, frame, col) in zip(axes, frames):
    for layer, y in layer_y.items():
        cnt = len(frame[layer])
        ax.barh(y, cnt, color=col, alpha=0.85, height=0.6)
        ax.text(cnt + 0.08, y, "  ".join(
            [f"({i+1})" for i in range(cnt)]),
            va="center", fontsize=8)
    ax.set_yticks(list(layer_y.values()))
    ax.set_yticklabels(list(layer_y.keys()))
    ax.set_xlabel("Number of elements")
    ax.set_xlim(0, 5)
    ax.set_title(title, fontsize=10)
    ax.invert_yaxis()
fig.suptitle("CLA: current vs alternative framing across 4 layers", fontsize=11)
fig.tight_layout()
plt.show()


## (B) 層間の支持関係マッピング

litany（表層）の各項目は、それを支える system・worldview・myth の要素によって生み出されている。ここでは、litany 項目が深層のどの要素に支えられているかを支持関係行列として表現し、ヒートマップで可視化する。CLA の主張は「表層の litany だけを書き換えても、それを支える深層のmyth/worldview が温存されていれば元の litany が再生産される」という点にある。litany だけ置換した場合と myth まで置換した場合とで、代替未来が整合的に持続するかを簡単な整合性指標で比較する。

In [ ]:
# --- 支持関係: litany 項目が depth(system/worldview/myth)要素に支えられる度合い ---
# 行 = litany 項目, 列 = 深層要素。1.0=強く支える, 0.5=部分的, 0=無関係。
DEEP_ELEMENTS = [
    ("myth", "機械の知性"),
    ("myth", "人間 vs 機械"),
    ("worldview", "効率至上主義"),
    ("worldview", "技術決定論"),
    ("worldview", "知能＝計算"),
    ("system", "労働市場の不安定化"),
    ("system", "プラットフォーム経済とデータ独占"),
]
LITANY_ITEMS = [
    "AIが仕事を奪う(見出し)",
    "AIが嘘をつく(速報)",
]

SUPPORT = np.array([
    # 機械の知性 人間vs機械 効率至上 技術決定論 知能=計算 労働市場 PF/データ独占
    [0.5,       1.0,      1.0,    1.0,      0.5,    1.0,    0.5],  # 仕事を奪う
    [1.0,       0.5,      0.5,    0.5,      1.0,    0.0,    0.5],  # 嘘をつく
])

print("[litany を支える深層要素の支持関係]")
print("-" * 70)
for i, lit in enumerate(LITANY_ITEMS):
    print(f"  litany: 「{lit}」を支える深層要素")
    supporters = [(DEEP_ELEMENTS[j], SUPPORT[i, j])
                  for j in range(len(DEEP_ELEMENTS)) if SUPPORT[i, j] > 0]
    for (layer, name), w in sorted(supporters, key=lambda x: -x[1]):
        print(f"      [{layer:>9}] {name}  (支持度 {w})")
    print()


In [ ]:
# --- 支持関係をヒートマップで可視化する ---
fig, ax = plt.subplots(figsize=(9, 3.4))
im = ax.imshow(SUPPORT, cmap="OrRd", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(DEEP_ELEMENTS)))
ax.set_xticklabels([f"{l[:4]}:{n}" for l, n in DEEP_ELEMENTS],
                   rotation=40, ha="right", fontsize=8)
ax.set_yticks(range(len(LITANY_ITEMS)))
ax.set_yticklabels([f"L{i+1}" for i in range(len(LITANY_ITEMS))], fontsize=9)
ax.set_xlabel("Deep-layer elements (system / worldview / myth)")
ax.set_ylabel("Litany items")
ax.set_title("CLA: how deep layers support the surface litany")
for i in range(len(LITANY_ITEMS)):
    for j in range(len(DEEP_ELEMENTS)):
        v = SUPPORT[i, j]
        if v > 0:
            ax.text(j, i, f"{v:.1f}", ha="center", va="center",
                    fontsize=8, color="white" if v > 0.6 else "black")
fig.colorbar(im, ax=ax, label="support strength")
fig.tight_layout()
plt.show()


In [ ]:
# --- 整合性指標: litany だけ置換 vs myth まで置換 ---
# 深層要素が「現状フレーミング由来」のまま残ると、元の litany を再生産する圧力になる。
# 代替 litany の持続可能性 = 1 - (温存された深層支持の重み平均)。
def alternative_consistency(replace_deep_layers=None):
    """replace_deep_layers: 置換する深層 layer 名の集合。
    置換された層の要素は代替フレーミング由来になり、元litanyを支えなくなる。
    """
    replace_deep_layers = replace_deep_layers or set()
    retained = []
    for j, (layer, _name) in enumerate(DEEP_ELEMENTS):
        retained.append(0.0 if layer in replace_deep_layers else 1.0)
    retained = np.array(retained)
    reproduction_pressure = (SUPPORT * retained).sum() / SUPPORT.sum()
    consistency = 1.0 - reproduction_pressure
    return reproduction_pressure, consistency

print("[代替未来の整合性: 置換範囲による違い]")
print("-" * 70)
scenarios = [
    ("litany のみ置換 (深層は据え置き)", set()),
    ("litany + system まで置換", {"system"}),
    ("litany + system + worldview まで置換", {"system", "worldview"}),
    ("4層すべて置換 (myth まで含む)", {"system", "worldview", "myth"}),
]
for name, rd in scenarios:
    pressure, cons = alternative_consistency(rd)
    bar = "#" * int(round(cons * 30))
    print(f"  {name}")
    print(f"      元litany再生産圧力 = {pressure:.2f}  /  "
          f"代替未来の整合性 = {cons:.2f}  {bar}")
print("-" * 70)
print("  -> 表層 litany だけを置換しても深層の myth/worldview が温存される限り")
print("     元の litany を再生産する圧力が残り、代替未来は整合的に持続しない。")
print("     myth まで置換して初めて整合性が最大化する = CLA の核心的主張。")


## (C) キーワード分類器の限界の定量化

冒頭のキーワード辞書による層別自動分類は、litany 層の語彙の偏りを見るには有効だが、worldview/myth の判定には使えない。ここではその限界を学習者が体験できるよう定量化する。(i) 2つの異なるキーワード辞書で同じ言説集合を分類し、層判定の不一致率を算出する。(ii) 本来 worldview/myth に属する言説が litany に誤分類される具体例を示す。litany は表層語彙で拾えるが、worldview/myth は解釈を要するため機械分類できない、という点を数値と具体例で確認する。

In [ ]:
# --- (i) 同じ言説を2つのキーワード辞書で分類し、層判定の不一致率を測る ---
# 辞書B は辞書A(=冒頭の LAYER_KEYWORDS)と同程度に「もっともらしい」別案。
# 効率/進歩 を system 寄り、知能/計算 を myth 寄りに振るなど、解釈の幅で割り当てが分かれる。
LAYER_KEYWORDS_B = {
    "litany": [
        "見出し", "速報", "ニュース", "炎上", "ランキング",
        "事故", "話題", "発表", "数字", "革命",
    ],
    "system": [
        "労働市場", "教育制度", "プラットフォーム", "規制", "政策",
        "投資", "制度", "市場", "効率", "進歩", "生産性",
    ],
    "worldview": [
        "技術決定論", "パラダイム", "前提", "価値観", "合理性",
        "世界観", "中立",
    ],
    "myth": [
        "機械の知性", "魔法", "神", "物語", "夢", "悪夢",
        "知性", "知能", "計算", "神話", "黙示録",
    ],
}

# キーワードが積み重ならない、現実的な語り口の言説（解釈の余地が大きい）
AMBIGUOUS_DISCOURSE = [
    "AIの進歩は止められないというのが多くの記事の前提になっている",
    "効率を上げることが社会全体の進歩だと、誰もが信じて疑わない",
    "AIは人間の知性を超える、という物語が革命のように語られる",
    "計算がすべてを解決するという価値観が、技術決定論を後押しする",
    "自動化の魔法が雇用構造を変えるという話題が投資を呼ぶ",
    "知能を機械に宿らせる夢が、研究の制度設計を方向づける",
]

mismatch = 0
print("[同一言説を2つの辞書で分類: 層判定の不一致]")
print("-" * 72)
for i, text in enumerate(AMBIGUOUS_DISCOURSE, start=1):
    la, _ = classify_discourse(text, LAYER_KEYWORDS)
    lb, _ = classify_discourse(text, LAYER_KEYWORDS_B)
    flag = "" if la == lb else "  <-- 不一致"
    if la != lb:
        mismatch += 1
    print(f"  言説{i}: 辞書A={la:<13} 辞書B={lb:<13}{flag}")
    print(f"         「{text}」")
rate = 100 * mismatch / len(AMBIGUOUS_DISCOURSE)
print("-" * 72)
print(f"  不一致 {mismatch}/{len(AMBIGUOUS_DISCOURSE)} 件  =  不一致率 {rate:.0f}%")
print("  -> 辞書をもっともらしく差し替えるだけで層判定が大きく揺れる。")
print("     とくに worldview/system/myth の境界は語彙では決まらず、")
print("     辞書設計者の解釈に判定が依存する。客観的な機械分類ではない。")


In [ ]:
# --- (ii) worldview/myth に属する言説が litany に誤分類される具体例 ---
# 「本来の層」は文の意味と前提を解釈して人手で付与した正解ラベル。
# 深層の言説でも、表層語彙(見出し・報道・統計など)を含むと litany に吸い寄せられる。
LABELED = [
    ("AIが仕事を奪うという見出しが連日報道されている", "litany"),
    ("「進歩は止められない」という前提を、どの報道も問い直さないまま見出しにする",
     "worldview"),
    ("AIが人類を救う神になるという物語が、誇大な速報や統計の見出しを生み続ける",
     "myth"),
    ("効率が最優先という価値観が当然視され、失業のニュースすら進歩の代償と語られる",
     "worldview"),
    ("自動化の魔法という夢が、新製品の発表や話題のランキングとして報道される",
     "myth"),
]

print("[本来 worldview/myth の言説がキーワード分類でどう判定されるか]")
print("-" * 72)
err = 0
for text, true_layer in LABELED:
    pred, scores = classify_discourse(text, LAYER_KEYWORDS)
    ok = (pred == true_layer)
    if not ok:
        err += 1
    mark = "正" if ok else "誤"
    print(f"  [{mark}] 本来={true_layer:<10} 機械判定={pred:<13}"
          f" scores={scores}")
    print(f"        「{text}」")
    if not ok:
        print(f"        -> 表層語彙に引きずられ、解釈を要する深層を litany に誤分類")
    print()
print("-" * 72)
print(f"  深層ラベル言説 {len(LABELED)} 件中 {err} 件を litany 等へ誤分類。")
print("  litany は表層語彙で拾えるが、worldview/myth は文の意味と前提を")
print("  解釈して初めて同定でき、キーワード一致では機械分類できない。")
print("  深層の同定には、機械分類ではなく解釈的な対話が要る(手法第6節)。")


## 未来デザイン論文での使われ方と結論への影響

未来デザイン論文において因果層別分析（CLA）は、技術をめぐる議論を四つの層——表層の公的言説、それを支える社会構造、構造を正当化する世界観、最深部の神話・隠喩——に分けて読み解くために用いられる。論文は表層の賛否や予測をそのまま分析対象とするのではなく、その議論を成り立たせている前提の地層を掘り下げ、最深層の神話を別の隠喩に置き換えた代替未来を提示する。そこで生まれる結論は、予測でも計画でもなく、問いの再フレーミングという型をとる。すなわち「この技術の影響を脅威として論じる問いの立て方そのものが、特定の神話に囚われている」という、議論の前提に向けられた批判的な言明が成果物となる。

この手法が結論に持ち込む規定力の中心は時間観にある。CLA は未来を過去の延長とも、確率的な分岐とも見ず、人々が無意識に共有する深層の物語によって縛られたものと捉える。ゆえに論文は「何が起きるか」を当てる作業を放棄し、「なぜそう語られるのか」を問い、隠喩を組み替えることで別の未来が思考可能になると主張する。境界設定の面では、分析者が選んだ言説資料の範囲と、どの言説をどの層に割り当てるかという解釈が、暴き出される神話の像を決める。

価値の所在という点で、CLA は価値判断を分析の出発点に置く。中立な記述を装わず、現状の支配的フレーミングを問い直すべきものとして扱う立場そのものが規範的である。深層に行くほど数値化になじまず、層の判定は分析者の解釈に依存するため、結論の妥当性は機械的な手続きではなく解釈の説得力に委ねられる。したがってこの手法を用いた論文は、表層のインパクト評価を再生産しない強みを持つ一方、複数の分析者による解釈の突き合わせを欠けば、結論が分析者自身の世界観の投影に終わる危うさを抱える。

## 発展課題

**課題A**: `CURRENT_FRAME` を、自分で集めたAI関連の言説資料（報道見出し・政策文書の抜粋・SNS投稿など）から再構成し、支配的な myth を同定せよ。そのうえで myth を別の隠喩に置換し、worldview → system → litany が連動して書き換わる `ALT_FRAME` を自分で構築せよ。

**課題B**: (B) の支持関係行列 `SUPPORT` を自分の言説資料に合わせて作り直し、litany だけを置換した場合と myth まで置換した場合とで代替未来の整合性がどう変わるかを確認せよ。表層だけの対症療法ではなぜ未来が変わらないのか、CLA の主張を自分の言葉で説明せよ。

**課題C**: (C) を踏まえ、キーワード分類が litany では機能しworldview/myth では機能しない理由を整理せよ。深層の同定に機械分類ではなく解釈的対話が要る、という CLA の方法論的立場を、自分が誤分類を見つけた具体例とともに論ぜよ。